### ENTREGA PARCIAL 2

#### 1. Identifica funciones necesarias con API

In [31]:
import requests
import uuid

def api_obtener_idiomas():
    # Endpoint para obtener los idiomas de traducción soportados
    url = "https://api.cognitive.microsofttranslator.com/languages?api-version=3.0&scope=translation"
    
    try:
        res = requests.get(url)
        res.raise_for_status()
        idiomas = res.json()['translation']
        
        return {datos['name']: iso for iso, datos in idiomas.items()}
    except Exception as e:
        print(f"Error al obtener idiomas: {e}")
        return {}

# B. Función de texto a texto
def api_traduccion_texto(texto, desde, a, clave, region):
    endpoint = "https://api.cognitive.microsofttranslator.com/translate"
    
    params = {
        'api-version': '3.0',
        'from': desde,
        'to': a
    }
    
    headers = {
        'Ocp-Apim-Subscription-Key': clave,
        'Ocp-Apim-Subscription-Region': region,
        'Content-type': 'application/json',
        'X-ClientTraceId': str(uuid.uuid4())
    }
    
    # El cuerpo debe ser una lista de diccionarios con la clave 'text'
    body = [{'text': texto}]
    
    try:
        response = requests.post(endpoint, params=params, headers=headers, json=body)
        response.raise_for_status()
        resultado = response.json()
        return resultado[0]['translations'][0]['text']
    except Exception as e:
        return f"Error en la traducción: {e}"

#### 2. Realiza una prueba con cada una de ellas

In [32]:
clave_traductor = "2HDx26ckfKhImGGCPmDnlX0LP9pjs8j7EFYA6GB76rxDnberSEEAJQQJ99CCAC5RqLJXJ3w3AAAbACOGixjt"
region = "westeurope"

# 1. Prueba de obtención de idiomas
diccionario_idiomas = api_obtener_idiomas()
if diccionario_idiomas:
    print(f"Se han cargado {len(diccionario_idiomas)} idiomas.")
    # Mostramos los primeros 10 para verificar
    for i, nombre in enumerate(list(diccionario_idiomas.keys())[:10]):
        print(f"{i+1}. {nombre} ({diccionario_idiomas[nombre]})")

print("-" * 30)

# 2. Prueba de traducción Texto a Texto
# Usamos 'es' (Español) y 'en' (Inglés) como prueba rápida
test_texto = "Bienvenido al proyecto traductor"
resultado = api_traduccion_texto(test_texto, "es", "en", clave_traductor, region)
print(f"Texto original: {test_texto}")
print(f"Texto traducido: {resultado}")

Se han cargado 137 idiomas.
1. Afrikaans (af)
2. Amharic (am)
3. Arabic (ar)
4. Assamese (as)
5. Azerbaijani (az)
6. Bashkir (ba)
7. Belarusian (be)
8. Bulgarian (bg)
9. Bhojpuri (bho)
10. Bangla (bn)
------------------------------
Texto original: Bienvenido al proyecto traductor
Texto traducido: Welcome to the translator project


#### 3. Identifica objetos y funciones necesarias con SDK.

Explícame para qué sirve cada objeto y función


In [27]:
import azure.cognitiveservices.speech as speechsdk

def sdk_audio_a_texto(idioma_entrada, clave, region):
    speech_config = speechsdk.SpeechConfig(subscription=clave, region=region)
    speech_config.speech_recognition_language = idioma_entrada
    
    audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
    recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)
    
    print(f"Escuchando... ({idioma_entrada})")
    result = recognizer.recognize_once_async().get()
    
    # Manejo de excepciones y errores específicos
    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        return result.text
    
    elif result.reason == speechsdk.ResultReason.NoMatch:
        print("Error: No se detectó sonido o el idioma no coincide.")
        
    elif result.reason == speechsdk.ResultReason.Canceled:
        detalles = result.cancellation_details
        print(f"Error: El proceso fue cancelado. Motivo: {detalles.reason}")
        
        # Identificación exacta del error técnico
        if detalles.reason == speechsdk.CancellationReason.Error:
            if "401" in detalles.error_details:
                print("Diagnóstico: Error 401. La clave o la región son incorrectas para el servicio de Speech.")
            elif "1006" in detalles.error_details:
                print("Diagnóstico: Error de conexión. Revisa tu conexión a internet.")
            elif "microphone" in detalles.error_details.lower():
                print("Diagnóstico: Error de hardware. Python no tiene acceso al micrófono.")
            else:
                print(f"Detalle técnico: {detalles.error_details}")
    
    return None

def sdk_texto_a_audio(texto, voz_idioma, clave, region):
    speech_config = speechsdk.SpeechConfig(subscription=clave, region=region)
    speech_config.speech_synthesis_voice_name = voz_idioma
    
    synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config)
    resultado = synthesizer.speak_text_async(texto).get()
    
    if resultado.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        return True
    else:
        print(f"Error en reproducción: {resultado.reason}")
        return False

#### 4. Realiza una prueba con cada una de ellas

In [33]:
# Credenciales para la prueba
clave_v = "FGR5ICm81XvfuSllSeTvNnaL5rBbVaneBlCag2PnSvGpOxVmKtM5JQQJ99CCAC5RqLJXJ3w3AAAYACOGLpG2"
region_v = "westeurope"

print("--- INICIANDO PRUEBAS DE DIAGNÓSTICO ---")

try:
    # 1. Prueba de captura
    texto_detectado = sdk_audio_a_texto("es-ES", clave_v, region_v)
    
    if texto_detectado:
        print(f"Capturado: {texto_detectado}")
        
        # 2. Prueba de salida
        print("Reproduciendo en altavoz...")
        sdk_texto_a_audio(f"Traducción lista: {texto_detectado}", "es-ES-AlvaroNeural", clave_v, region_v)
    else:
        print("La prueba de audio a texto no devolvió ningún resultado.")

except Exception as e:
    print(f"Error inesperado en la ejecución: {e}")

--- INICIANDO PRUEBAS DE DIAGNÓSTICO ---
Escuchando... (es-ES)
Capturado: Antes estaba pillado a TI hablando.
Reproduciendo en altavoz...
